In [39]:
import pandas as pd
import sqlite3
from pathlib import Path
import numpy as np

db_path = Path("../data/olist.db")
conn = sqlite3.connect(db_path)


def get_df(filename):
    path = Path(f"../sql/{filename}")
    text = path.read_text()
    df = pd.read_sql(text,conn)
    return df



In [3]:
print(pd.read_sql("SELECT SUM(items_value) FROM orders WHERE in_revenue = 1", conn))

get_df("01_revenue_by_category.sql")["revenue"].sum() 

get_df("01_revenue_by_category.sql")[get_df("01_revenue_by_category.sql")["category"] == "unknown"]


   SUM(items_value)
0       13221498.11


,category,revenue,n_of_orders,n_of_items,percentage_of_total
20,unknown,170726.63,1392,1537,1.29


In [4]:
get_df("01_revenue_by_category.sql").head(2)

,category,revenue,n_of_orders,n_of_items,percentage_of_total
0,health_beauty,1233131.72,8647,9465,9.33
1,watches_gifts,1166176.98,5495,5859,8.82


sql/01_revenue_by_category.sql calculates the revenue of each product category, counting only delivered orders (in_revenue = 1). Revenue comes from order_items.price (item level), not from the order totals, because joining the orders table to the items would repeat each order once per item.

The query returns 74 categories, including unknown (products with no category, which still have sales). The sum of the category revenues is 13221498.11, identical to the total revenue in the orders table, so no rows were lost or duplicated.

-The top category is health_beauty, with 1,233,131.72 (9.33% of the revenue), followed by watches_gifts (8.82%) and bed_bath_table (7.74%).

-The top 5 categories account for about 39.8% of the revenue, so the revenue is spread across many categories instead of depending on a few.

-health_beauty is first in revenue but not in number of orders: bed_bath_table has more (9,272 vs. 8,647), which suggests that health_beauty has a higher average price per order.

In [5]:
df_02 = get_df("02_monthly_orders_revenue.sql")
df_02

,month_of_purchase,n_of_orders,revenue,average_ticket,revenue_variation
0,2017-01,750,111798.36,149.06,NaN
1,2017-02,1653,234223.40,141.70,109.51
2,2017-03,2546,359198.85,141.08,53.36
3,2017-04,2303,340669.68,147.92,-5.16
4,2017-05,3546,489338.25,138.00,43.64
5,2017-06,3135,421923.37,134.58,-13.78
6,2017-07,3872,481604.52,124.38,14.15
7,2017-08,4193,554699.70,132.29,15.18
8,2017-09,4150,607399.67,146.36,9.50
9,2017-10,4478,648247.65,144.76,6.73


In [6]:
max_revenue_month = df_02.loc[[df_02["revenue"].idxmax()], "month_of_purchase"]
max_n_of_orders = df_02.loc[[df_02["n_of_orders"].idxmax()], "month_of_purchase"]
pd.DataFrame({"best_month_revenue": max_revenue_month, "best_month_n_of_orders": max_n_of_orders})


,best_month_revenue,best_month_n_of_orders
10,2017-11,2017-11


In [7]:
print(f"total orders: {df_02['n_of_orders'].sum()}")
print(f"total revenue: {round(df_02['revenue'].sum(),2)}")
pd.read_sql("SELECT SUM(items_value) FROM orders WHERE in_revenue = 1 AND month_of_purchase >= '2017-01'",conn)


total orders: 96211
total revenue: 13181027.13


,SUM(items_value)
0,13181027.13


In [8]:
first_semestre_2017 = df_02[df_02["month_of_purchase"] < "2017-07"]["revenue"].sum()
first_semestre_2018 = df_02[(df_02["month_of_purchase"] > "2017-12") & (df_02["month_of_purchase"] < "2018-07")]["revenue"].sum()
print(f"Revenue from 1º semestre 2017 to 1º semestre 2018: {round((first_semestre_2018 - first_semestre_2017)/(first_semestre_2017)*100, 2)}")

print(first_semestre_2018, first_semestre_2017)

Revenue from 1º semestre 2017 to 1º semestre 2018: 181.61
5511595.0200000005 1957151.9100000001


sql/02_monthly_orders_revenue.sql returns, for each month, the number of orders, the revenue, the average ticket and the revenue variation compared with the previous month. Only delivered orders are counted (in_revenue = 1), and the period is 2017-01 to 2018-08.

I excluded 2016 because it has only three months (September, October and December), with 267 orders and one missing month (November). A month-over-month comparison with such a small base gives meaningless values (for example, +29,777%).

-The peak is November 2017, with 7,289 orders and 987,765 in revenue. It is a possible Black Friday effect, but the data does not confirm it.

-In the first half of 2018 the revenue was about 2.8 times the first half of 2017 (+181.61%), but from 2018 the monthly revenue stays between about 826 thousand and 978 thousand, so the growth stops.

-The average ticket changes little (between 124 and 149), so the revenue growth comes from the number of orders and not from higher prices.

-The sum of the monthly revenue (13,181,027.13) and of the orders (96,211) matches the totals of the orders table for the same period.

In [9]:
df_03 = get_df("03_delay_vs_review.sql")
df_03

,delay_group,n_of_orders,average_review_score,p_of_total,pct_low_score
0,Is delayed,6379,2.27,6.68,62.41
1,Is not delayed,89182,4.29,93.32,9.25


In [10]:
print(pd.read_sql("SELECT SUM(in_review_analysis) FROM orders WHERE month_of_purchase >= '2017-01'", conn))
print(df_03["n_of_orders"].sum())

   SUM(in_review_analysis)
0                    95561
95561


In [11]:
df_03_1 = get_df("03b_delay_vs_review.sql")
df_03_1

,delay_group,n_of_orders,average_review_score,p_of_total,pct_low_score
0,On time,89182,4.29,93.32,9.25
1,1-3 days,1851,3.29,1.94,32.14
2,4-7 days,1748,2.10,1.83,67.68
3,8-14 days,1447,1.67,1.51,80.10
4,15+ days,1333,1.72,1.39,78.32


sql/03_delay_vs_review.sql and sql/03b_delay_vs_review.sql compare the review scores of delayed and on-time orders. The scope is in_review_analysis = 1 (orders with a review and a defined delay) from 2017-01 on, the same period as the monthly analysis: 95,561 orders, about 6.7% of them delayed. 

My hypothesis was that more delay means worse reviews. The data confirms it only partly:

-Delayed orders have an average score of 2.27, against 4.29 for on-time orders (2.02 points lower). 62.41% of the delayed orders got a score of 1 or 2, against 9.25% of the on-time ones.

-Even a short delay hurts: the average drops to 3.29 with 1 to 3 days of delay, to 2.10 with 4 to 7 days and to 1.67 with 8 to 14 days. After that it stops falling (1.72 with 15 or more days), so the steepest drop is between the first and the second bucket.

-The groups with 4 or more days of delay are small (about 1.4% to 1.8% of the orders each), but 68% to 80% of their reviews are 1 or 2 stars.


In [12]:
df_04 = get_df("04_delivery_by_state.sql")

In [13]:
print(df_04["states"].count())
print(pd.read_sql("SELECT COUNT(DISTINCT customer_state) as n_of_states FROM customers", conn))

27
   n_of_states
0           27


In [14]:
print(df_04["n_of_orders"].sum())
print(pd.read_sql("SELECT COUNT(order_id) as total_orders FROM orders WHERE in_delivery_analysis = 1",conn))

96476
   total_orders
0         96476


In [15]:
top3_higher_time_to_delivery = df_04.head(3)
top3_higher_time_to_delivery

,states,n_of_orders,median_days,average_days,pct_delayed,slowness_rank
0,RR,41,25.0,29.0,12.20,1
1,AM,145,25.0,26.0,2.76,2
2,AP,67,24.0,26.7,2.99,3


In [16]:
top3_lower_time_to_delivery = df_04.tail(3)
top3_lower_time_to_delivery

,states,n_of_orders,median_days,average_days,pct_delayed,slowness_rank
24,MG,11355,10.0,11.5,4.58,25
25,PR,4923,10.0,11.5,4.04,26
26,SP,40495,7.0,8.3,4.49,27


In [17]:
df_04

,states,n_of_orders,median_days,average_days,pct_delayed,slowness_rank
0,RR,41,25.0,29.0,12.20,1
1,AM,145,25.0,26.0,2.76,2
2,AP,67,24.0,26.7,2.99,3
3,AL,397,22.0,24.0,21.41,4
4,PA,946,21.0,23.3,11.21,5
5,MA,717,19.0,21.1,17.43,6
6,SE,335,18.0,21.0,15.22,7
7,CE,1279,18.0,20.8,13.76,8
8,AC,80,18.0,20.6,3.75,9
9,PB,517,18.0,20.0,10.44,10


sql/04_delivery_by_state.sql returns, for each of the 27 states, the number of orders, the median and average delivery time (in days), the percentage of delayed orders and a slowness rank based on the median. The scope is in_delivery_analysis = 1 (96,476 delivered orders), and customers are joined with a one-to-one join, because customer_id is unique.

-RR is the slowest state (median of 25 days) and SP the fastest (7 days). The three slowest states (RR, AM and AP) are in the north of the country, but they have few orders (41, 145 and 67), so their values are less reliable. Among the states with at least 300 orders, the slowest are AL (22 days), PA (21) and MA (19).

-The ranking by median is almost identical to the ranking by average (the same three states are on top, and only AM and AP swap places), so the conclusion does not depend on the measure. The average is 1 to 4 days higher than the median in every state, which means that a few very late deliveries pull the average up.

-Delivery time and delay rate do not always agree. AM, AP, AC and RO are among the slowest (17 to 25 days) but have less than 4% of delayed orders, while AL, MA and SE have 15% to 21% of delayed orders with 18 to 22 days.

-SP concentrates about 42% of the orders, so the national figures are mostly driven by the fastest state.

The data shows where deliveries take longer, not why. A possible explanation is the distance between the seller and the customer, but I did not test it.

In [37]:
df_05 = get_df("05_seller_ranking.sql")
df_05b = get_df("05b_seller_concentration.sql")

In [40]:
total_revenue = pd.read_sql("SELECT SUM(items_value) AS total FROM orders WHERE in_revenue = 1", conn)["total"].iloc[0]
print(np.isclose(df_05["revenue"].sum(), total_revenue, atol=0.5))

True


In [44]:
top10 = df_05["revenue"].head(10).sum() / df_05["revenue"].sum() * 100
print(np.isclose(top10, df_05b["top_10_sellers_pct"].iloc[0], atol=0.05))
print((df_05["accumulated_prc"] >= 50).idxmax() + 1 == df_05b["sellers_for_50pct"].iloc[0])   
print((df_05["accumulated_prc"] >= 80).idxmax() + 1 == df_05b["sellers_for_80pct"].iloc[0])

print(df_05b[["top_10_sellers_pct", "top_1pct_sellers_pct", "top_10pct_sellers_pct", "top_20pct_sellers_pct"]].T)

True
True
True
                           0
top_10_sellers_pct     13.27
top_1pct_sellers_pct   25.51
top_10pct_sellers_pct  67.11
top_20pct_sellers_pct  82.29


In [46]:
df_05
df_05b

,n_sellers,total_revenue,top_10_sellers_pct,top_1pct_sellers_pct,top_10pct_sellers_pct,top_20pct_sellers_pct,sellers_for_50pct,sellers_for_80pct
0,2970,13221498.11,13.27,25.51,67.11,82.29,127,533


sql/05a_seller_ranking.sql returns one row per seller, with its revenue, number of orders, share of the total revenue and cumulative share; sql/05b_seller_concentration.sql summarizes the concentration in a single row. The scope is in_revenue = 1, and the revenue of each seller is the sum of order_items.price (without freight). I used the item prices and not `orders.items_value, because an order can contain items from several sellers, and the order value cannot be assigned to any one of them: it would also be repeated once per item.

The sum of the seller revenues is 13,221,498.11, identical to the total revenue of the orders table.

-The top 10 sellers account for 13.27% of the revenue, the top 1% for 25.51%, the top 10% for 67.11% and the top 20% for 82.29%.

-127 sellers are enough to reach 50% of the revenue, and 533 to reach 80%.

 This shows how concentrated the revenue is, but not why. 